<a href="https://colab.research.google.com/github/Nwaduito/Tech-project/blob/main/AI_Mavericks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Data Cleaning**

In [ ]:
from google.colab import files
uploaded = files.upload()

print('='*50)
print('Step 1. Import our libraries')
print('='*50)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported!')

print('='*50)
print('Step 2. Load our dataset')
print('='*50)
df_raw = pd.read_csv('amazon.csv')
print(f" Dataset loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print()
print("Columns:", df_raw.columns.tolist())
print()
print(df_raw.head(3))

print('='*50)
print('Step 3. Explore our raw dataset')
print('='*50)
print("=== DATA OVERVIEW ===")
print(f"Total rows: {len(df_raw):,}")
print(f"Unique products: {df_raw['product_id'].nunique():,}")
print(f"Unique categories: {df_raw['category'].nunique():,}")
print()
print("Missing values:")
print(df_raw.isnull().sum())
print()
print("Sample categories (raw):")
print(df_raw['category'].value_counts().head(5))

print('='*50)
print('Step 4. Clean and prepare the data')
print('='*50)
# WHY WE NEED TO CLEAN THIS DATASET
# Before we can build any recommendation model, the data must be
# in a format Python can actually work with. The amazon.csv dataset
# has several issues we discovered during exploration:
#
# ISSUE 1 — Price columns have a rupee symbol and commas
#   Raw value looks like: "₹1,099"
#   Python cannot do maths on a string like that.
#   We need to strip the symbol and commas so it becomes: 1099.0
#
# ISSUE 2 — discount_percentage has a % symbol
#   Raw value looks like: "64%"
#   Again, Python cannot use this as a number until we clean it.
#
# ISSUE 3 — rating has 2 null (missing) values
#   We cannot train a model on missing data, so we drop those rows.
#
# ISSUE 4 — rating_count has commas in the numbers
#   Raw value looks like: "24,269"
#   We remove the comma so Python reads it as 24269 (an integer).
#
# ISSUE 5 — category is a pipe-separated hierarchy
#   Raw value looks like: "Electronics|Mobiles|Smartphones"
#   For our recommender we only need the top level (Electronics),
#   so we split on the pipe and take the first part.
#
# ISSUE 6 — user_id has MULTIPLE users in one cell
#   Raw value looks like: "USER1,USER2,USER3,USER4"
#   This happened because multiple users reviewed the same product.
#   Collaborative filtering needs ONE user per row, so we must
#   explode this column — splitting each user into its own row.
#   This gives us a proper interactions table.
df = df_raw.copy()
# Clean the price columns
# The regex '[\u20b9,]' matches two characters in one pass:
#   \u20b9 is the unicode code for the rupee symbol ₹
#   ,  is the thousands separator comma
# We remove both in one step, then convert the result to a number.
# errors='coerce' means: if a value still cannot be converted
# (e.g. some unexpected text), turn it into NaN instead of crashing.
df['discounted_price'] = (
    df['discounted_price']
    .astype(str)
    .str.replace('[\u20b9,]', '', regex=True)
    .str.strip()
)
df['discounted_price'] = pd.to_numeric(df['discounted_price'], errors='coerce')

df['actual_price'] = (
    df['actual_price']
    .astype(str)
    .str.replace('[\u20b9,]', '', regex=True)
    .str.strip()
)
df['actual_price'] = pd.to_numeric(df['actual_price'], errors='coerce')
# Clean discount percentage
df['discount_percentage'] = (
    df['discount_percentage']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.strip()
)
df['discount_percentage'] = pd.to_numeric(df['discount_percentage'],
errors='coerce')
# Clean rating
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
# Clean rating count
df['rating_count'] = (
    df['rating_count']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['rating_count'] = pd.to_numeric(df['rating_count'], errors='coerce').fillna(0).astype(int)
# Extract main category (first part before the pipe) e.g "Electronics|Mobiles|Smartphones" becomes "Electronics"
df['main_category'] = df['category'].astype(str).str.split('|').str[0].str.strip()
# Extract sub-category
df['sub_category'] = df['category'].astype(str).apply(
    lambda x: x.split('|')[1].strip() if len(x.split('|')) > 1 else x.split('|')[0].strip()
)
# Drop rows missing fields
# We only drop rows where product_id, product_name, rating or
# main_category is missing — these are fields our models depend on.
# We also drop duplicate products so each product_id appears once.
before = len(df)
df = df.dropna(subset=['product_id', 'product_name', 'rating', 'main_category'])
df = df.drop_duplicates(subset='product_id')
after = len(df)
print(f"\n Cleaning done! Rows: {before} → {after} (removed {before - after})")

# Build interactions table
# The user_id column in the raw data stores multiple users per row,
# separated by commas. For example one row might have 8 user IDs.
# Collaborative filtering requires one user-product pair per row,
# so we need to "explode" this column.
#
# Step 1: Split the comma-separated string into a Python list
# Step 2: explode() turns each item in the list into its own row
# Step 3: Strip whitespace left over from the split
# Step 4: Filter out any user_id shorter than 6 characters.
#         Real Amazon user IDs are long alphanumeric strings.
#         Anything very short is an empty string or junk produced
#         by the split and should be removed.
interactions = df[['product_id', 'product_name', 'main_category', 'rating', 'user_id']].copy()
interactions['user_id'] = interactions['user_id'].astype(str).str.split(',')
interactions = interactions.explode('user_id')
interactions['user_id'] = interactions['user_id'].str.strip()
interactions = interactions[interactions['user_id'].str.len() > 5]  # remove blanks

print(f"\n Interactions table built!")
print(f"   Total interactions: {len(interactions):,}")
print(f"   Unique users:       {interactions['user_id'].nunique():,}")
print(f"   Unique products:    {interactions['product_id'].nunique():,}")

print("="*50)
# Step 5. Data Cleaning and EDA charts
print("="*50)
print("\n CLEAN DATASET SUMMARY")
print("=" * 45)
print(f"Products:         {df['product_id'].nunique():,}")
print(f"Main categories:  {df['main_category'].nunique():,}")
print(f"Price range:      Rs.{df['discounted_price'].min():.0f} - Rs.{df['discounted_price'].max():,.0f}")
print(f"Rating range:     {df['rating'].min()} - {df['rating'].max()}")
print(f"Avg rating:       {df['rating'].mean():.2f}")
print()
print("Products per main category:")
print(df['main_category'].value_counts())

# Charts
# Chart 1 shows the top 10 most reviewed products.
# We use rating_count (number of reviews) not rating (the star score)
# because a product with many reviews is more trustworthy as "popular".
#
# Chart 2 shows how many products fall into each main category.
# This helps us understand which categories dominate our dataset
# and informs how well the category-based fallback will perform.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_reviewed = df.nlargest(10, 'rating_count')[['product_name', 'rating_count']].copy()
top_reviewed['product_name'] = top_reviewed['product_name'].str[:35] + '...'
axes[0].barh(top_reviewed['product_name'], top_reviewed['rating_count'], color='purple')
axes[0].set_title('Top 10 Most Reviewed Products')
axes[0].set_xlabel('Review Count')
axes[0].invert_yaxis()

cat_counts = df['main_category'].value_counts()
axes[1].bar(cat_counts.index, cat_counts.values, color='teal')
axes[1].set_title('Products per Main Category')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Product Count')

plt.tight_layout()
plt.savefig('eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("EDA chart saved: eda_charts.png")

print('='*50)
print('Step 6. Save the cleaned datasets')
print('='*50)

# --- Save the cleaned products table ---
# This is the main cleaned dataset with one row per product.
# All prices are now numbers, categories are split into
# main_category and sub_category, and nulls have been removed.
# We save it as amazon_cleaned.csv so any team member can load
# this directly without having to re-run the cleaning steps.
df.to_csv('amazon_cleaned.csv', index=False)
print(f"amazon_cleaned.csv saved — {len(df):,} rows, {len(df.columns)} columns")

# --- Save the interactions table ---
# This is the exploded user-product table needed for
# collaborative filtering. Each row is one user-product pair.
# We save it separately so the CF model can load it directly.
interactions.to_csv('amazon_interactions.csv', index=False)
print(f"amazon_interactions.csv saved — {len(interactions):,} rows")

print()
print("Both files are now in your project folder.")
print("You can load them in any other script using:")
print("  df = pd.read_csv('amazon_cleaned.csv')")
print("  interactions = pd.read_csv('amazon_interactions.csv')")
import os
print(os.getcwd())
